# ChainGuard - Demo

**A Multi-Agent, Explainable and Blockchain-Audited Cryptocurrency Crash Early-Warning System**

> Neural networks predict patterns. Agents investigate and reason over evidence.
> Blockchain makes predictions auditable.

This notebook runs top to bottom against the **frozen dataset**. Every number below
is computed by the pipeline - nothing is hardcoded (Rule 12.2).

Prerequisites: `bolt build`, `bolt train`, `bolt evaluate --all --report` and
`bolt explain --model xgb --consistency` have been run.


## 0. Setup and dataset integrity

The first thing a panel should see: the dataset we are about to use is *the same
one* the reported results came from. The data card records a SHA-256; recompute it.


In [ ]:
import hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from bolt.config import load_config
from bolt.logging_setup import setup_logging
setup_logging({'level': 'ERROR'})

cfg = load_config('../config/default.yaml', repo_root='..')
panel_path = cfg.path('processed') / 'panel.parquet'
digest = hashlib.sha256(panel_path.read_bytes()).hexdigest()
card = (cfg.path('processed') / 'DATA_CARD.md').read_text(encoding='utf-8')
recorded = [ln for ln in card.splitlines() if 'SHA-256' in ln][0]

print('frozen dataset :', panel_path.name)
print('sha256         :', digest)
print('data card says :', recorded.split(chr(96))[1])
print()
print('MATCH' if digest in recorded else 'MISMATCH - dataset changed since the card')


## 1. The data

10 assets, 2020-2025. The panel is deliberately **ragged**: assets are absent
before their inception rather than back-filled, because inventing history for an
asset that did not yet trade is exactly what Guard 1 forbids.


In [ ]:
panel = pd.read_parquet(panel_path)
print('panel:', panel.shape)

dates = panel.index.get_level_values('date')
rows = []
for asset, block in panel.groupby(level='asset'):
    d = block.index.get_level_values('date')
    rows.append({'asset': asset, 'rows': len(block),
                 'first': d.min().date(), 'last': d.max().date(),
                 'positive_labels': int(block['label'].sum(skipna=True))})
pd.DataFrame(rows).set_index('asset')


## 2. G1 - what counts as a crash, and why

The base paper predicts a **pin-bar reversal**: a candlestick pattern, i.e. a
trading signal. We predict a **20% drawdown within 14 days**: a systemic event.
Different target, different research question.

The sensitivity table is what makes that choice defensible rather than arbitrary -
it shows what every *other* choice would have produced.


In [ ]:
sens = pd.read_csv(cfg.path('tables') / 'label_sensitivity.csv')
sens


Note which configurations **miss the USDC de-peg**: a 7-day horizon is too short
and a 25% threshold too deep. The default (0.20, 14) is the tightest setting that
still covers all four crisis episodes, and `assert_episodes_covered()` raises if
that ever stops being true.


## 3. Risk timeline with the crises marked


In [ ]:
btc = panel.xs('BTC', level='asset')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(btc.index, btc['close'], lw=1.1, color='#1f3b57')
ax1.set_yscale('log'); ax1.set_ylabel('BTC close (log scale)')
ax1.set_title('BTC with configured crisis episodes and positive crash labels')

for ep in cfg.episodes:
    for ax in (ax1, ax2):
        ax.axvspan(pd.Timestamp(ep.start, tz='UTC'),
                   pd.Timestamp(ep.end, tz='UTC'), color='crimson', alpha=0.20)
    ax1.text(pd.Timestamp(ep.start, tz='UTC'), btc['close'].max() * 0.9,
             ep.name, rotation=90, fontsize=7, va='top', color='crimson')

ax2.fill_between(btc.index, btc['label'].fillna(0), step='mid',
                 color='#c0392b', alpha=0.7)
ax2.set_ylabel('crash label'); ax2.set_ylim(0, 1.2); ax2.set_xlabel('date')
plt.tight_layout(); plt.show()


## 4. G3 - the seven-model comparison

Leak-free walk-forward with a 45-day embargo. Every model sees identical inputs
through `flatten_windows()`; if they did not, the comparison would not be fair.


In [ ]:
comparison = pd.read_csv(cfg.path('tables') / 'model_comparison.csv', index_col=0)
comparison


In [ ]:
per_fold = pd.read_csv(cfg.path('tables') / 'model_comparison_per_fold.csv')
base = per_fold.groupby('fold')['pr_auc_baseline'].first()
print('random-classifier PR-AUC per fold - the number every result must beat:')
print(base.round(3).to_string())
print()
per_fold.pivot(index='model', columns='fold', values='pr_auc').round(3)


### Reading the result honestly

**Random Forest wins. The deep models do not.** The one-line volatility rule ties
the from-scratch LSTM and beats the GRU and the MLP.

Spec Rule 12.7 is explicit: *if a simpler model wins, say so; do not tune the deep
model until it wins.* So: on this target, with this feature set, recurrence is not
earning its complexity.

The best model reaches roughly 2x the random baseline, and fold variance is large -
a model that works in 2022 may not work in 2023.

The base paper's F1 of 0.703 is on pin-bar reversals, a far more frequent event.
The numbers are not comparable and we do not claim they are. **Any PR-AUC near 0.95
on this task would be evidence of leakage, not skill.**


## 5. Lead time - the metric that matters operationally

A warning that arrives after the crash has started is not a warning.


In [ ]:
lead = pd.read_csv(cfg.path('tables') / 'lead_time.csv')
lead.pivot(index='episode', columns='model', values='lead_time_days')


`NaN` means **no sustained warning fired** before that episode. A sustained
crossing requires at least 2 consecutive days above threshold, so single-day noise
does not count as a warning. These failures are reported per episode and never
averaged away.


## 6. G4 - do the same drivers explain every crisis?

The headline research question. The base paper applies SHAP but never tests whether
its explanations are *stable*. An explanation that changes completely between
Terra/LUNA and FTX is not a mechanism; it is a per-episode description dressed up
as one.


In [ ]:
consistency = pd.read_csv(cfg.path('tables') / 'attribution_consistency.csv')
spearman = pd.read_csv(cfg.path('tables') / 'attribution_spearman_matrix.csv',
                       index_col=0)
print('stability index:', round(float(consistency['stability_index'].iloc[0]), 3))
print()
print(consistency['interpretation'].iloc[0])
print()
spearman.round(3)


In [ ]:
from IPython.display import Image, display
for name in ['attribution_consistency_heatmap.png', 'top_features_per_episode.png']:
    path = cfg.path('figures') / name
    if path.exists():
        display(Image(str(path)))


The interpretation thresholds are fixed in `interpret_stability()` **before** any
result is computed, so the conclusion cannot be retrofitted to whatever number came
out. High consistency would be evidence of a generalisable crash signature; low
consistency is an important negative result bounding how far any
explanation-based crash warning generalises. Both are publishable; only a
fabricated one is not.


## 7. The agent chain - one explained warning

Run on the eve of the FTX collapse. Each agent reports independently, the
orchestrator aggregates, the **skeptic challenges the conclusion**, and only then
is a decision made.


In [ ]:
from bolt.agents.pipeline import ChainGuardPipeline, build_context
from bolt.commands import load_models

try:
    models, scaler = load_models(cfg)
    print('loaded models:', sorted(models), '\n')
except FileNotFoundError as exc:
    models, scaler = {}, None
    print('no trained models -', exc, '\n')

ctx = build_context(cfg, panel, 'BTC', pd.Timestamp('2022-11-05'),
                    list(cfg.feature_columns))
result = ChainGuardPipeline(cfg, models, scaler).run(ctx, commit=False)
print(result.render())


> ### WARNING - these dates are IN-SAMPLE
>
> `bolt train` fits the deployment models on every window whose label resolves
> before 2024-11-17, so 2022-11-05 is **inside the training data**. What this cell
> demonstrates is that the agent chain wires together correctly and produces a
> coherent, explained decision - *not* that the system predicted FTX.
>
> Presenting an in-sample hit as foresight is precisely the hindsight fitting that
> G5 exists to make impossible. The honest out-of-sample evidence is the
> walk-forward table in section 4.

Note the **Skeptic Agent**. It is not decoration: its counter-evidence lowers the
final confidence, and when confidence falls below the floor the Decision Agent
returns `INSUFFICIENT_EVIDENCE` rather than issuing a warning it cannot support.

Note also which agents mark themselves `DEGRADED` and why - the On-Chain agent
declares that four of its five inputs are documented proxies, and the News agent
declares that event identification is not implemented.


## 8. The historical analogue

*When did the market last look like this, and what happened next?*

`exclude_days` is load-bearing: consecutive 30-day windows share 29 days of data,
so without it the nearest neighbours of today are yesterday and the day before.
A window from last week is not a historical precedent.


In [ ]:
from bolt.commands import load_windows
from bolt.explain.analogue import nearest_historical_analogue
from bolt.windows import flatten_windows

X, y, meta = load_windows(cfg)
primary = (meta['asset'] == 'BTC').to_numpy()
flat = flatten_windows(X[primary])
sub_meta = meta[primary].reset_index(drop=True)

ends = pd.to_datetime(sub_meta['window_end'], utc=True)
target = int(np.flatnonzero(ends <= pd.Timestamp('2022-11-05', tz='UTC'))[-1])

analogues = nearest_historical_analogue(
    flat[target], flat, sub_meta, prices=panel['close'], k=3,
    exclude_days=90, as_of=pd.Timestamp('2022-11-05', tz='UTC'))

for a in analogues:
    o = a['what_happened_next_30d']
    outcome = (f"{o['max_drawdown']:.1%} drawdown" if o.get('available')
               else 'outcome unavailable')
    print(f"{a['date']}  similarity {a['similarity']:.3f}  -> next 30d: {outcome}")


## 9. G5 - the on-chain proof

The strongest and most original claim. Every predictive paper in this field - the
base paper included - asks the reader to trust a backtest that could in principle
have been tuned after the fact, and nothing in the published record distinguishes
genuine foresight from hindsight fitting. A cryptographic commitment on a public
ledger is the only mechanism that resolves this.


In [ ]:
payload = result.commitment.prediction.canonical()
print('canonical payload - the exact bytes a verifier hashes:')
print(payload[:420].decode(), '...')
print()
print('prediction id :', result.commitment.prediction.prediction_id)
print('sha-256       :', result.commitment.digest)
print('payload file  :', result.commitment.payload_path)


In [ ]:
# Determinism: shuffling the dict must not change the digest.
from bolt.chain.payload import canonical_payload, digest_hex

original = result.commitment.prediction.to_dict()
shuffled = {k: original[k] for k in reversed(list(original))}

print('key order      :', list(original)[:3], 'vs', list(shuffled)[:3])
print('same digest    :',
      digest_hex(canonical_payload(original)) == digest_hex(canonical_payload(shuffled)))
print()
print('A verifier recomputing this hash on another machine gets the same result,')
print('which is what makes a mismatch mean tampering rather than noise.')


### Independent verification

```bash
bolt verify --payload outputs/predictions/<id>.json --id <prediction_id>
```

`chain/verify.py` re-implements canonicalisation from the standard library and is
**tested to import nothing from the pipeline it verifies** - verification that
depends on the system it verifies proves nothing.

The contract's overwrite revert is what the whole claim rests on, and it is tested
against a real compiled EVM in `tests/test_contract.py`: nobody, *including the
original committer*, can replace a commitment once it is made.


## Summary

| Contribution | Status |
|---|---|
| G1 economically meaningful target | done, with a published sensitivity table |
| G2 cross-asset contagion features | done, correlation graph + eigenvector centrality |
| G3 seven-model fair comparison | done - RF wins, deep models do not, reported as found |
| G4 attribution consistency across crises | done, thresholds fixed in advance |
| G5 on-chain commitment | contract and payload tested; testnet deploy pending |

**What has actually been built:** a reproducible pipeline from public APIs to a
leak-free evaluated model comparison, with cross-episode attribution consistency,
a multi-agent evidence layer that challenges its own predictions, and a
cryptographic commitment scheme that makes the track record auditable.

**What has not:** the LLM narrative backend and news event identification
(Phase 7), and a live testnet deployment, which needs a funded Amoy key.
